# NASDAQ-100 end-to-end demo

Run all cells with the Python environment from the README. The notebook uses
checked-in synthetic fixtures and trains the project's GAT-LSTM on one short
walk-forward fold for a one-trading-day prediction. The outputs illustrate the
pipeline on synthetic data. No downloads or model artifacts are required.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "configs/gat_lstm.yaml").is_file()), None)
if repo is None:
    raise RuntimeError("Open the notebook from inside the nasdaq100-news-graph checkout")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.core.config import load_config
from src.core.io import read_table
from src.core.paths import FIXTURES_DIR
from src.models.panel import build_panel
from src.models.train import folds_for, train_fold

tables = {
    name: read_table(name, FIXTURES_DIR / f"{name}.parquet")
    for name in ("prices", "labels", "news_features", "edges")
}
print({name: len(table) for name, table in tables.items()})


## Features and walk-forward split

The shared model panel builds technical indicators from prices and aligns
news features and labels by trading day and ticker. The training code scales
features using the training fold and holds out validation and test days with
a five-session embargo.


In [ ]:
panel = build_panel(tables["prices"], tables["labels"], tables["news_features"])
split = {
    "period": {"start": "2021-01-01", "end": "2021-12-31"},
    "train_days": 30,
    "test_days": 10,
    "step_days": 10,
    "expanding": False,
    "embargo_days": 5,
}
fold = folds_for(panel, split)[0]

print(f"Train: {fold.train[0].date()} to {fold.train[-1].date()} ({len(fold.train)} sessions)")
print(f"Test:  {fold.test[0].date()} to {fold.test[-1].date()} ({len(fold.test)} sessions)")


## Train and predict

Train a small GAT-LSTM with price and news features and all three edge types.
Use the validation-selected threshold to turn each held-out probability into
an up/down prediction.


In [ ]:
config = load_config("gat_lstm.yaml")
config.update(window_days=5, embargo_days=split["embargo_days"])
config["features"].update(embedding_dim=4)
config["encoder"].update(hidden=8)
config["gat"].update(heads=2)
config["train"].update(epochs=3, early_stopping_metric="loss")

horizon = 1
predictions, info = train_fold(panel, tables["edges"], fold, horizon, config)
expected = panel.rows(fold.test, horizon)
assert not predictions.empty
assert predictions[["date", "ticker"]].equals(expected[["date", "ticker"]])
assert predictions["p_up"].between(0, 1).all()

print(f"Held-out predictions: {len(predictions)}")
print(f"Validation threshold: {info['threshold']:.2f}")
predictions.assign(predicted_up=predictions["p_up"] >= info["threshold"])[
    ["date", "ticker", "p_up", "predicted_up", "y_true"]
].head(10)
